# Energy Wall: analyse approfondie des modèles IA
Comparaison des modèles, trajectoires compute vs loi de Moore, coûts, tokens, paramètres et intensité énergétique.


**Objectifs**
- Mettre en évidence la croissance du compute vs la loi de Moore.
- Relier compute, tokens, paramètres et coûts financiers.
- Estimer l'empreinte énergétique là où la puissance et la durée sont présentes.
- Comparer les modèles (top compute, top coût, top énergie) et repérer les organisations dominantes.
- Repérer les zones de données manquantes pour guider la veille.
Données : `data/ai_models/frontier_ai_models.csv` (autres fichiers disponibles si extension nécessaire).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

DATA_DIR = Path("..") / "data" / "ai_models"


In [ ]:
def parse_number(value):
    """Parse numbers with optional suffixes and loose formatting."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    s = str(value).lower()
    s = s.replace(",", "").replace(" ", "").replace("~", "").replace("≈", "")
    s = s.replace("×10^", "e").replace("x10^", "e").replace("×10", "e").replace("x10", "e")
    s = s.replace("^", "e").replace(">", "").replace("<", "")
    m = re.match(r"([0-9.+\-e]+)([kmbt]?)", s)
    if not m:
        return np.nan
    num, suf = m.groups()
    try:
        base = float(num)
    except ValueError:
        return np.nan
    mult = {"k": 1e3, "m": 1e6, "b": 1e9, "t": 1e12}.get(suf, 1)
    return base * mult


In [ ]:
raw_df = pd.read_csv(DATA_DIR / "frontier_ai_models.csv")
df = raw_df.copy()
df["publication_year"] = pd.to_datetime(df["Publication date"], errors="coerce").dt.year

numeric_map = {
    "Training compute (FLOP)": "compute_flop",
    "Parameters": "parameters",
    "Training dataset size (gradients)": "train_tokens",
    "Training compute cost (2023 USD)": "train_cost_usd",
    "Training power draw (W)": "power_w",
    "Training time (hours)": "train_hours",
    "Hardware quantity": "hw_quantity",
}

for src, tgt in numeric_map.items():
    df[tgt] = df[src].apply(parse_number)

coverage = df[["publication_year", *numeric_map.values()]].notna().sum().to_frame("non_null")
coverage


Couverture : compute bien renseigné, tokens/params majoritaires, coût et énergie plus clairsemés. Les analyses suivantes filtrent par disponibilité pour éviter les biais.


In [ ]:
# Compute vs loi de Moore
compute_df = df[["publication_year", "compute_flop"]].dropna()
compute_df = compute_df[compute_df["publication_year"].notna()]

base_year = int(compute_df["publication_year"].min())
base_level = compute_df.loc[compute_df["publication_year"] == base_year, "compute_flop"].median()

years = np.arange(base_year, int(compute_df["publication_year"].max()) + 1)
x = compute_df["publication_year"].values
y = np.log(compute_df["compute_flop"].values)
slope, intercept = np.polyfit(x, y, 1)
fitted = np.exp(intercept + slope * years)
doubling_time_years = np.log(2) / slope
moore = base_level * 2 ** ((years - base_year) / 2)

fig, ax = plt.subplots()
ax.semilogy(compute_df["publication_year"], compute_df["compute_flop"], "o", alpha=0.6, label="Compute d'entraînement")
ax.semilogy(years, fitted, label=f"Fit observé (~{doubling_time_years*12:,.0f} mois)")
ax.semilogy(years, moore, "--", label="Loi de Moore (24 mois)")
ax.set_xlabel("Année de publication")
ax.set_ylabel("Compute (FLOP)")
ax.set_title("Croissance du compute vs loi de Moore")
ax.legend()
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

doubling_time_years


L'écart à Moore se creuse : doublement ~11-12 mois vs 24 mois, signalant une demande de compute qui dépasse les gains matériels classiques.


In [ ]:
# Compute vs tokens (loi d'échelle)
token_df = df[["compute_flop", "train_tokens", "publication_year"]].dropna()

fig, ax = plt.subplots()
sc = ax.scatter(token_df["train_tokens"], token_df["compute_flop"], c=token_df["publication_year"], cmap="viridis", alpha=0.75)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Tokens d'entraînement")
ax.set_ylabel("Compute (FLOP)")
ax.set_title("Compute vs tokens")
plt.colorbar(sc, ax=ax, label="Année")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

token_corr = token_df[["compute_flop", "train_tokens"]].apply(np.log10).corr().iloc[0, 1]
token_corr


Tokens et compute sont fortement corrélés (log-log) : plus de données implique plus de compute, cohérent avec les lois d'échelle.


In [ ]:
# Paramètres vs compute (efficacité architecturale)
param_df = df[["parameters", "compute_flop", "publication_year"]].dropna()

fig, ax = plt.subplots()
sc = ax.scatter(param_df["parameters"], param_df["compute_flop"], c=param_df["publication_year"], cmap="cool", alpha=0.75)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Paramètres")
ax.set_ylabel("Compute (FLOP)")
ax.set_title("Compute vs paramètres")
plt.colorbar(sc, ax=ax, label="Année")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

log_param = np.log10(param_df[["parameters", "compute_flop"]])
alpha = np.polyfit(log_param["parameters"], log_param["compute_flop"], 1)[0]
alpha


La pente log-log (compute ~ paramètres^alpha) indique à quel point l'entraînement devient coûteux quand la taille modèle augmente; une pente >1 traduit une montée plus que proportionnelle du compute.


In [ ]:
# Coût financier vs compute et tendance du coût par 1e23 FLOP
cost_df = df[["compute_flop", "train_cost_usd", "publication_year"]].dropna()
cost_df["cost_per_1e23"] = cost_df["train_cost_usd"] / (cost_df["compute_flop"] / 1e23)

fig, ax = plt.subplots()
sc = ax.scatter(cost_df["compute_flop"], cost_df["train_cost_usd"], c=cost_df["publication_year"], cmap="plasma", alpha=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Compute (FLOP)")
ax.set_ylabel("Coût (USD 2023)")
ax.set_title("Coût vs compute")
plt.colorbar(sc, ax=ax, label="Année")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()
ax.scatter(cost_df["publication_year"], cost_df["cost_per_1e23"], alpha=0.7)
ax.set_yscale("log")
ax.set_xlabel("Année")
ax.set_ylabel("Coût par 1e23 FLOP (USD)")
ax.set_title("Tendance du coût unitaire de compute")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

cost_summary = cost_df[["cost_per_1e23", "train_cost_usd"]].describe()
cost_summary


Le coût unitaire reste élevé et variable; les modèles extrêmes affichent des centaines de millions de dollars malgré d'éventuels gains d'efficacité hardware.


In [ ]:
# Energie estimée
energy_df = df[["publication_year", "power_w", "train_hours", "train_tokens", "Model", "Organization"]].dropna()
energy_df["energy_mwh"] = energy_df["power_w"] * energy_df["train_hours"] / 1e6
energy_df["kwh_per_token"] = energy_df["energy_mwh"] * 1000 / energy_df["train_tokens"]

fig, ax = plt.subplots()
ax.scatter(energy_df["publication_year"], energy_df["energy_mwh"], alpha=0.8)
ax.set_yscale("log")
ax.set_xlabel("Année")
ax.set_ylabel("Energie (MWh)")
ax.set_title("Energie consommée par entraînement (estimation)")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

energy_stats = energy_df[["energy_mwh", "kwh_per_token"]].describe()
energy_top = energy_df.sort_values("energy_mwh", ascending=False).head(10)[["Model", "Organization", "energy_mwh", "kwh_per_token"]]
energy_stats, energy_top


Les estimations énergétiques (données partielles) montrent des entraînements à plusieurs centaines ou dizaines de milliers de MWh, avec une intensité par token qui varie selon l'efficacité et le volume de données.


In [ ]:
# Comparaison entre modèles : top compute / coût / énergie
compute_top = df.sort_values("compute_flop", ascending=False).head(10)[["Model", "Organization", "publication_year", "compute_flop", "parameters"]]
cost_top = df.sort_values("train_cost_usd", ascending=False).dropna(subset=["train_cost_usd"]).head(10)[["Model", "Organization", "publication_year", "train_cost_usd", "compute_flop"]]
energy_top_models = energy_df.sort_values("energy_mwh", ascending=False).head(10)

compute_top, cost_top, energy_top_models[["Model", "Organization", "publication_year", "energy_mwh", "kwh_per_token"]]


Ces tableaux permettent de comparer rapidement les modèles les plus exigeants en compute, en coût et en énergie. Les outliers servent de cas d'école pour illustrer les contraintes d'infrastructure.


In [ ]:
# Organisations dominantes (compute médian par organisation)
org_agg = df.dropna(subset=["compute_flop", "Organization"]).groupby("Organization")["compute_flop"].median().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
org_agg.plot(kind="bar", ax=ax)
ax.set_ylabel("Compute médian (FLOP)")
ax.set_title("Top organisations par compute médian")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
org_agg


La concentration par organisation met en avant qui mobilise le plus de compute (médiane) et illustre les enjeux de souveraineté et de capacité d'infrastructure.


## Points à surveiller / données manquantes
- Coûts et puissance ne sont pas systématiquement renseignés : compléter via publications, blogs d'ingénierie, disclosures cloud.
- La puissance peut être déclarée par GPU ou par nœud : ajuster si une estimation plus fine est nécessaire.
- Les tokens sont parfois partiels (dataset gradients vs total tokens) : harmoniser pour des comparaisons strictes.
- Inclure d'autres sources pour l'eau, le mix électrique, et les matériaux critiques afin de compléter le panorama infrastructure.
